# KDEllipsPy: Synthetic Simulation Example for Google Colab

This notebook demonstrates how to install KDEllipsPy and run a synthetic earthquake simulation using the Axitra forward engine.

### 1. Installation

We clone the repository and install the package. The installation will automatically compile the Axitra Fortran binaries and Python wrappers.

In [ ]:
!git clone https://github.com/alexvillarroel/KDEllipsPy.git
%cd KDEllipsPy
# 1. Install build tools and the 'Stable Zone' stack (NumPy 1.x, PyMC 5.1x)
!pip install meson ninja "numpy>=1.26.4,<2.0.0" "scipy<1.14.0" "pandas>=2.2.2" "pymc>=5.10.0,<5.20.0" "arviz>=0.18.0,<0.20.0" --quiet
# 2. Install the package (this will compile Axitra)
!pip install .

### 2. Setup and Imports

In [ ]:
from kdellipspy import ConfigParser, AxitraForwardModel
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# Define the input control file (using an existing example)
input_file = "inversions/calama2020/input.ctl"
cfg = ConfigParser(input_file)

print(f"Loaded configuration for event: {cfg.source_position.event_name}")

### 3. Run Synthetic Simulation

We initialize the forward model and simulate an elliptical slip patch.

In [ ]:
# Initialize Forward Model
# It automatically finds the compiled axitra binaries inside the package
fm = AxitraForwardModel(input_file)

# Define elliptical model parameters:
# [axis1_km, axis2_km, rotation_rad, pos_n, pos_t, dmax_m, vr_kms]
model_params = [10.0, 7.0, 0.25 * np.pi, 0.5, 0.5, 3.0, 2.8]

print("Running Axitra simulation...")
synthetics, time = fm.simulate_ellipse(model_params)

print(f"Simulation finished!")
print(f"Synthetics shape: {synthetics.shape} (stations, components, samples)")

### 4. Visualization

We plot the 3-component seismograms for the first few stations.

In [ ]:
n_stations_to_plot = min(3, synthetics.shape[0])
fig, axes = plt.subplots(n_stations_to_plot, 3, figsize=(15, 3 * n_stations_to_plot), sharex=True)

components = ["North", "East", "Vertical"]

for i in range(n_stations_to_plot):
    station_name = cfg.station_params.stations[i].name
    for j in range(3):
        ax = axes[i, j] if n_stations_to_plot > 1 else axes[j]
        ax.plot(time, synthetics[i, j, :], color="tab:blue", lw=1.5)
        if i == 0: ax.set_title(components[j])
        if j == 0: ax.set_ylabel(f"{station_name}
Velocity")
        ax.grid(True, linestyle="--", alpha=0.7)

plt.xlabel("Time (s)")
plt.suptitle("Synthetic Seismograms - KDEllipsPy Forward Model", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()